In [1]:
import pandas as pd
import sqlite3
import os
import re
import unicodedata

def normalize_column(col):

    col = col.replace("\xa0", " ")
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("ascii")
    col = col.lower()
    col = re.sub(r"[^\w\s]", "", col)
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"_+", "_", col)

    return col.strip("_").upper()


directory = "files"

file_names = [
    r"C:\Users\Yuan\Desktop\git_hub\cepel_axia_eletrobras\files\xlsx\Composição do Stack Tecnológico de Dados no Cepel_Parte 0_Identificação.xlsx",
    r"C:\Users\Yuan\Desktop\git_hub\cepel_axia_eletrobras\files\xlsx\Composição do Stack Tecnológico de Dados no Cepel_Parte 1_Geral.xlsx",
    r"C:\Users\Yuan\Desktop\git_hub\cepel_axia_eletrobras\files\xlsx\Composição do Stack Tecnológico de Dados no Cepel_Parte 2_Avançada.xlsx",
    r"C:\Users\Yuan\Desktop\git_hub\cepel_axia_eletrobras\files\xlsx\Composição do Stack Tecnológico de Dados no Cepel_Parte 3_Interface DevOps.xlsx"
]

db_path = "cepel_data.db"
conn = sqlite3.connect(db_path)

print(f"Connected to SQLite database at {db_path}")

# carregar excel -> sqlite
for file_name in file_names:

    file_path = os.path.join(directory, file_name)

    df = pd.read_excel(file_path)
    df.columns = [normalize_column(c) for c in df.columns]

    table_name = file_name.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_").split(".")[0]
    table_name = os.path.splitext(os.path.basename(file_name))[0].replace(" ", "_")
    df.to_sql(table_name, conn, if_exists="replace", index=False)

    print(f"Loaded {file_name} -> table {table_name}")


# pegar todas as tabelas
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("\nTables in database:")
for table in tables:
    print(table[0])

Connected to SQLite database at cepel_data.db
Loaded C:\Users\Yuan\Desktop\git_hub\cepel_axia_eletrobras\files\xlsx\Composição do Stack Tecnológico de Dados no Cepel_Parte 0_Identificação.xlsx -> table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação
Loaded C:\Users\Yuan\Desktop\git_hub\cepel_axia_eletrobras\files\xlsx\Composição do Stack Tecnológico de Dados no Cepel_Parte 1_Geral.xlsx -> table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_1_Geral
Loaded C:\Users\Yuan\Desktop\git_hub\cepel_axia_eletrobras\files\xlsx\Composição do Stack Tecnológico de Dados no Cepel_Parte 2_Avançada.xlsx -> table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_2_Avançada
Loaded C:\Users\Yuan\Desktop\git_hub\cepel_axia_eletrobras\files\xlsx\Composição do Stack Tecnológico de Dados no Cepel_Parte 3_Interface DevOps.xlsx -> table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps

Tables in database:
Composição_do_Stack_Tecnológico_de_Dados_

In [22]:
cursor.close()
conn.close()

In [18]:
# salvar todas as tabelas em CSV
for table in tables:
    table_name = table[0]
    print(f"\nExporting table {table_name} to CSV...")


Exporting table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação to CSV...

Exporting table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_1_Geral to CSV...

Exporting table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_2_Avançada to CSV...

Exporting table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps to CSV...


In [30]:
# salvar todas as tabelas em CSV
for table in tables:

    table_name = table[0]

    df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)

    file_output = f"{table_name}.csv"

    df.to_csv(
        file_output,
        index=False,
        encoding="utf-8-sig",
        sep=";"
    )

    print(f"Tabela {table_name} salva em {file_output}")


print("\nTodas as tabelas foram exportadas com sucesso!")

Tabela Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação salva em Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação.csv
Tabela Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_1_Geral salva em Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_1_Geral.csv
Tabela Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_2_Avançada salva em Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_2_Avançada.csv
Tabela Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps salva em Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps.csv

Todas as tabelas foram exportadas com sucesso!


In [8]:
import sqlite3
import pandas as pd
from pathlib import Path

# Dynamic path: go up one level from scripts/ to reach the project root, then into files/
# db_path = Path(__file__).parent.parent / "files" / "cepel_data.db"

# In a Jupyter notebook, use this instead:
db_path = Path.cwd().parent / "scripts" / "cepel_data.db"
conn = sqlite3.connect(str(db_path))
# Show full column content (no truncation with "...")
pd.set_option('display.max_colwidth', None)
# List all tables in the SQLite database
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)
print(tables)

                                                                         name
0     Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação
1             Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_1_Geral
2          Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_2_Avançada
3  Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps


In [9]:
table1 = pd.read_sql_query('''SELECT * FROM Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação''', conn)

In [10]:
#table1.count()

In [11]:
# Carrega um recorte da tabela de identificação dos respondentes
table1 = pd.read_sql_query('''SELECT * FROM Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação''', conn)

import pandas as pd

def SENIORIDADE_TEMPO_DE_CASA_NO_CEPEL(exp):
    if exp in ['Menos de 1 ano', '1-2 anos']:
        return 'Junior'
    elif exp in ['3-5 anos']:
        return 'Pleno'
    elif exp in ['6-10 anos','11-15 anos', '16-20 anos', '21-25 anos']:
        return 'Senior'
    else:
        return 'Outro'

table1['SENIORIDADE_TEMPO_DE_CASA_NO_CEPEL'] = table1['FAVOR_APONTAR_O_SEU_TEMPO_DE_CASA_NO_CEPEL'].apply(SENIORIDADE_TEMPO_DE_CASA_NO_CEPEL)



def HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS(exp):
    if exp in ['Menos de 1 ano', '1-2 anos']:
        return 'Junior'
    elif exp in ['3-5 anos']:
        return 'Pleno'
    elif exp in ['5-10 anos','Mais de 10 anos']:
        return 'Senior'
    else:
        return 'Outro'

table1['SENIORIDADE_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS'] = table1['HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS'].apply(HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS)



def HA_QUANTO_TEMPO_VOCE_JA_TRABALHA_COM_DADOS_GOVERNANCA_ENGENHARIA_CIENCIA_OU_ANALISE_DE_DADOS(exp):
    if exp in ['Menos de 1 ano', '1-2 anos']:
        return 'Junior'
    elif exp in ['3-5 anos']:
        return 'Pleno'
    elif exp in ['5-10 anos', 'Mais de 10 anos']:
        return 'Senior'
    else:
        return 'Outro'

table1['SENIORIDADE_TEMPO_VOCE_JA_TRABALHA_COM_DADOS_GOVERNANCA_ENGENHARIA_CIENCIA_OU_ANALISE_DE_DADOS'] = table1['HA_QUANTO_TEMPO_VOCE_JA_TRABALHA_COM_DADOS_GOVERNANCA_ENGENHARIA_CIENCIA_OU_ANALISE_DE_DADOS'].apply(HA_QUANTO_TEMPO_VOCE_JA_TRABALHA_COM_DADOS_GOVERNANCA_ENGENHARIA_CIENCIA_OU_ANALISE_DE_DADOS)



#table1[['SENIORIDADE_TEMPO_DE_CASA_NO_CEPEL','FAVOR_APONTAR_O_SEU_TEMPO_DE_CASA_NO_CEPEL']]
#table1[['SENIORIDADE_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS','HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS']]

In [12]:
# table1['FAVOR_APONTAR_SUA_FAIXA_ETARIA'].unique()
# ['25-34 anos', '35-44 anos', '45-54 anos', '18-24 anos'],não representa a serionidade

# table1['FAVOR_APONTAR_SEU_MAIOR_NIVEL_DE_FORMACAO_EDUCACIONAL'].unique()
# ['Especialização','Graduação','Mestrado / MBA','Doutorado','Ensino médio / profissionalizante']


In [13]:

#table1.count()

In [14]:
# Salvando de forma básica

table1.to_csv(f'doc_1_.csv', index=False)
print(f"Arquivo salvo como doc_1.csv")

Arquivo salvo como doc_1.csv


In [32]:
#resultado_ = df.groupby("NAME")["ID"].count().reset_index(name="CONTAGEM")
#resultado_ = table1.groupby("NAME","EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO").size().reset_index(name="CONTAGEM")
#resultado_

resultado = table1.groupby(["NAME","EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO"]).size().reset_index(name="CONTAGEM")

In [34]:
resultado.to_csv(f'doc_0.csv', index=False)
print(f"Arquivo salvo como doc_0.csv")

Arquivo salvo como doc_0.csv


In [8]:
#table1.head(5)

In [15]:
table4 = pd.read_sql_query('''SELECT * FROM Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps''', conn)

In [16]:
print(table4.columns.to_list())

['ID', 'START_TIME', 'COMPLETION_TIME', 'EMAIL', 'NAME', 'LAST_MODIFIED_TIME', 'WINDOWS_NT_SERVER', 'WINDOWS_1011', 'LINUX_DEBIAN_UBUNTU_MINT_KALI', 'LINUX_ARCH_MANJARO_ENDEAVOUROS', 'LINUX_RED_HAT_FEDORA_CENTOS_ROCKY', 'LINUX_SUSE', 'LINUX_ALPINE', 'MACOS', 'CHROME_OS', 'ANDROID', 'IOS', 'OUTRO6', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_SISTEMA_OPERACIONAL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_SISTEMAS_OPERACIONAIS_LHE_SAO_PRIORITARIOS', 'SSH', 'RDP_WINDOWS', 'VNC_MULTIPLATAFORMA', 'WSL_WINDOWS_SUBSYSTEM_FOR_LINUX', 'VMWARE', 'VIRTUALBOX', 'PROXMOX', 'DOCKER', 'PODMAN', 'SINGULARITY', 'DOCKERCOMPOSE', 'DOCKERSWARM', 'KUBERNETES', 'OUTRO7', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_MODELO_DE_VIRTUALIZACAO_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_MODELOS_DE_VIRTUALIZACAO_OU_DE_ACESSO_VIRTUAL_A_RECURSOS_LHE_SAO_PRIORITARIOS', 'SQL', 'PYTHON', 'R', 'SCALA', 'JAVA', 'JAVASCRIPT', 'TYPESCRIPT', 'SHE

In [17]:
table4 = pd.read_sql_query('''SELECT * FROM Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps''', conn)

grupos = {

    # ================= SISTEMA OPERACIONAL =================
    "SISTEMA_OPERACIONAL": [
        "WINDOWS_NT_SERVER",
        "WINDOWS_1011",
        "LINUX_DEBIAN_UBUNTU_MINT_KALI",
        "LINUX_ARCH_MANJARO_ENDEAVOUROS",
        "LINUX_RED_HAT_FEDORA_CENTOS_ROCKY",
        "LINUX_SUSE",
        "LINUX_ALPINE",
        "MACOS",
        "CHROME_OS",
        "ANDROID",
        "IOS",
        "OUTRO6"
    ],
    "SISTEMA_OPERACIONAL_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_SISTEMA_OPERACIONAL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],
    "SISTEMA_OPERACIONAL_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_SISTEMAS_OPERACIONAIS_LHE_SAO_PRIORITARIOS"
    ],

    # ================= VIRTUALIZAÇÃO =================
    "VIRTUALIZACAO_E_ACESSO": [
        "SSH","RDP_WINDOWS","VNC_MULTIPLATAFORMA","WSL_WINDOWS_SUBSYSTEM_FOR_LINUX",
        "VMWARE","VIRTUALBOX","PROXMOX","DOCKER","PODMAN","SINGULARITY",
        "DOCKERCOMPOSE","DOCKERSWARM","KUBERNETES","OUTRO7"
    ],
    "VIRTUALIZACAO_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_MODELO_DE_VIRTUALIZACAO_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],
    "VIRTUALIZACAO_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_MODELOS_DE_VIRTUALIZACAO_OU_DE_ACESSO_VIRTUAL_A_RECURSOS_LHE_SAO_PRIORITARIOS"
    ],

    # ================= LINGUAGENS =================
    "LINGUAGEM_PROGRAMACAO": [
        "SQL","PYTHON","R","SCALA","JAVA","JAVASCRIPT","TYPESCRIPT","SHELL_SCRIPT",
        "PHP","RUBY","GO","RUST","C_E_C","C","JULIA","FORTRAN","MATLAB","LABVIEW2","OUTRA"
    ],
    "LINGUAGEM_OUTRA": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRA_LINGUAGEM_DE_PROGRAMACAO_VOLTADA_A_DADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],
    "LINGUAGEM_PRIORITARIA": [
        "QUAIS_DAS_SEGUINTES_LINGUAGENS_DE_PROGRAMACAO_VOLTADAS_A_DADOS_LHE_SAO_PRIORITARIAS"
    ],

    # ================= IDEs =================
    "IDES_E_FERRAMENTAS_DEV": [
        "MS_VISUAL_STUDIO","VS_CODE","CURSOR","ANTIGRAVITY","ECLIPSE","INTELLIJ_IDEA",
        "NETBEAMS","PYCHARM","SUBLIME","VIM","EMACS","LABVIEW","CODER_CDE",
        "JUPYTER_LABNOTEBOOK","GITHUB_CODESPACES","KAGGLE_NOTEBOOKS",
        "GOOGLE_COLABORATORY","BACKSTAGE_IDP","OPSLEVEL","COMPASS","PORT","OUTRO10"
    ],
    "IDES_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_DESENVOLVIMENTO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],
    "IDES_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_DESENVOLVIMENTO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS"
    ],

    # ================= VERSIONAMENTO =================
    "CONTROLE_DE_VERSAO": [
        "CVS_CONCURRENT_VERSION_SYSTEM","SVN_APACHE_SUBVERSION","MERCURIAL",
        "GIT_GITHUB","GIT_GITLAB","GIT_BITBUCKET","GIT_LFS_LARGE_FILE_STORAGE",
        "DVC_DATA_VERSION_CONTROL","LAKEFS","OUTRO2"
    ],
    "CONTROLE_VERSAO_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_CONTROLE_DE_VERSAO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],
    "CONTROLE_VERSAO_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_CONTROLE_DE_VERSAO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS"
    ],

    # ================= INFRA =================
    "INFRAESTRUTURA_COMO_CODIGO": [
        "TERRAFORM","OPENTOFU","PULUMI","CROSSPLANE","ANSIBLE",
        "PUPPET","CHEF","SALT","NORNIR","VAGRANT","OUTRO5"
    ],
    "INFRA_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_CONFIGURACAO_DE_RECURSOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],
    "INFRA_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_CONFIGURACAO_DE_RECURSOS_LHE_SAO_PRIORITARIOS"
    ],

    # ================= CI/CD =================
    "CI_CD": [
        "JENKINS","TRAVIS_CI","CIRCLE_CI","GITHUB_ACTIONS","GITLAB_CICD",
        "TEAMCITY","BITBUCKET_PIEPELINES","FLUXCD","ARGOCD","OCTOPUSDEPLOY","OUTRO3"
    ],
    "CI_CD_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_CICD_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],
    "CI_CD_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_CICD_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS"
    ],

    # ================= METODOLOGIAS =================
    "METODOLOGIAS_GESTAO": [
        "PMBOK","SWEBOK","DMBOK","KANBAN","SCRUM","EXTREME_PROGRAMMING_XP","SAFE","OUTRO"
    ],
    "METODOLOGIA_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_COMPENDIO_DE_BOAS_PRATICAS_PARA_GESTAO_DE_PROJETOS_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],
    "METODOLOGIA_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_COMPENDIOS_DE_BOAS_PRATICAS_PARA_GESTAO_DE_PROJETOS_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS"
    ],

    # ================= BOAS PRÁTICAS =================
    "BOAS_PRATICAS_ENGENHARIA": [
        "DEVOPS","DEVSECOPS_OWASP_TOP_10","DATAOPS","MLOPS",
        "CLEAN_ARCHITECTURE_E_CLEAN_CODE","DDD_DOMAINDRIVEN_DESIGN","DESIGN_PATTERNS",
        "BDD_BEHAVIORDRIVEN_DESIGN","TDD_TESTDRIVEN_DEVELOPMENT","CRISPDM",
        "PRINCIPIOS_SOLID","PRINCIPIOS_KISS","PRINCIPIOS_YAGNI",
        "REFACTORING","THE_TWELVEFACTOR_APP","OUTRO9"
    ],
    "BOAS_PRATICAS_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_COMPENDIO_DE_BOAS_PRATICAS_EM_DESENVOLVIMENTO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],
    "BOAS_PRATICAS_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_COMPENDIOS_DE_BOAS_PRATICAS_EM_DESENVOLVIMENTO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS"
    ],

    # ================= GESTÃO =================
    "FERRAMENTAS_GESTAO_PROJETOS": [
        "SCOPI","BSC_DESIGNER","TRELLO","PLANNER","CLICKUP","JIRA",
        "LINEAR","GLPI","MANTIS","BUGZILLA","OUTRO4"
    ],
    "GESTAO_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_APOIO_METODOLOGICO_AO_DESENVOLVIMENTO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],
    "GESTAO_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_APOIO_METODOLOGICO_AO_DESENVOLVIMENTO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS"
    ],

    # ================= DOCUMENTAÇÃO =================
    "DOCUMENTACAO_COLABORACAO": [
        "MS_SHAREPOINT","CONFLUENCE","NOTION","MONDAYCOM","MIRO","MS_WHITEBOARD",
        "SLACK","MS_TEAMS","DOXYGEN","DRAWIO","PLANTUML","SWAGGER","CKAN",
        "RESTRUCTUREDTEXT_REST","SPHINX","MKDOCS","READ_THE_DOCS",
        "KEEP_A_CHANGELOG","SCRIBE","OBS_STUDIO","OUTRO8"
    ],
    "DOCUMENTACAO_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_DOCUMENTACAO_DE_PROJETOSSOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],
    "DOCUMENTACAO_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_DOCUMENTACAO_DE_PROJETOSSOFTWAREDADOS_LHE_SAO_PRIORITARIOS"
    ]
}

id_cols = ["ID", "EMAIL", "NAME"]

dfs = []

# grupos.items() -> it will return a view object that displays a list of a dictionary's key-value tuple pairs.
# dict_items([('SISTEMA_OPERACIONAL', ['WINDOWS_NT_SERVER', 'WINDOWS_1011', 'LINUX_DEBIAN_UBUNTU_MINT_KALI', 'LINUX_A....

for categoria, cols in grupos.items():

    df_temp = table4.melt(
        id_vars=id_cols,
        value_vars=cols,
        var_name="TECNOLOGIA",
        value_name="RESPOSTA"
    )

    df_temp["CATEGORIA"] = categoria

    dfs.append(df_temp)

df_final = pd.concat(dfs, ignore_index=True)

# Replacing Regex r"OUTRO\d+"
df_final["TECNOLOGIA"] = df_final["TECNOLOGIA"].str.replace(r"OUTRO\d+", "OUTRO", regex=True)

In [18]:
resultado = (
    table1
    .merge(
        df_final.drop(columns=["ID","EMAIL"], errors="ignore"),
        on="NAME",
        how="left"
    )
)

In [19]:
'''
# Calcuting total values for each category, department, seniority and technology

dfs_resultados = {}

for categoria in grupos.keys():

    df_result = (
        resultado[resultado["CATEGORIA"] == categoria]
        .groupby(["EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO","HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS","TECNOLOGIA"])
        .size()
        .reset_index(name="TOTAL")
    )
    df_result

    #import pandas as pd

    # Salvando de forma básica
    #df_result.to_csv(f'{categoria}.csv', index=False)

    #print(f"Arquivo salvo como '{categoria}.csv'")

'''

'\n# Calcuting total values for each category, department, seniority and technology\n\ndfs_resultados = {}\n\nfor categoria in grupos.keys():\n\n    df_result = (\n        resultado[resultado["CATEGORIA"] == categoria]\n        .groupby(["EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO","HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS","TECNOLOGIA"])\n        .size()\n        .reset_index(name="TOTAL")\n    )\n    df_result\n\n    #import pandas as pd\n\n    # Salvando de forma básica\n    #df_result.to_csv(f\'{categoria}.csv\', index=False)\n\n    #print(f"Arquivo salvo como \'{categoria}.csv\'")\n\n'

In [20]:
# resultado

In [21]:
# Calcuting total values for each category, department, seniority and technology

#df_result = resultado.groupby(["NAME","EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO"]).size().reset_index(name="TOTAL")
#df_result = resultado.groupby(["SENIORIDADE_TEMPO_DE_CASA_NO_CEPEL",
#                               "SENIORIDADE_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS",
#                               "EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO",]).size().reset_index(name="TOTAL")


In [41]:
#resultado.columns

In [22]:
# Salvando de forma básica

resultado.to_csv(f'doc_4_.csv', index=False)
print(f"Arquivo salvo como doc_4.csv")

Arquivo salvo como doc_4.csv


# Analise relacionado a stack de dados

In [43]:
import sqlite3
import pandas as pd
from pathlib import Path

# Dynamic path: go up one level from scripts/ to reach the project root, then into files/
# db_path = Path(__file__).parent.parent / "files" / "cepel_data.db"

# In a Jupyter notebook, use this instead:
db_path = Path.cwd().parent / "scripts" / "cepel_data.db"
conn = sqlite3.connect(str(db_path))

# List all tables in the SQLite database
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)
print(tables)

                                                                         name
0     Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação
1             Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_1_Geral
2          Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_2_Avançada
3  Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps


In [44]:
# Carrega um recorte da tabela de identificação dos respondentes
table3 = pd.read_sql_query('''SELECT * FROM Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_2_Avançada''', conn)
table3.columns

Index(['ID', 'START_TIME', 'COMPLETION_TIME', 'EMAIL', 'NAME',
       'LAST_MODIFIED_TIME', 'BLOCK_STORAGE', 'FILE_STORAGE', 'OBJECT_STORAGE',
       'QUAIS_DOS_SEGUINTES_FORMATOS_PARA_ARMAZENAMENTO_E_GESTAO_DE_DADOS_LHE_SAO_PRIORITARIOS',
       ...
       'ELK_STACK_ELASTIC_LOGSTASH_KIBANA', 'GRAPHITE', 'FLUENTD', 'SPLUNK',
       'NEW_RELIC', 'DATADOG', 'AMBARI', 'OUTRO2',
       'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_MONITORAMENTO_DE_DADOSRECURSOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
       'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_MONITORAMENTO_DE_DADOSRECURSOS_LHE_SAO_PRIORITARIOS'],
      dtype='str', length=116)

In [45]:
print(table3.columns.tolist())

['ID', 'START_TIME', 'COMPLETION_TIME', 'EMAIL', 'NAME', 'LAST_MODIFIED_TIME', 'BLOCK_STORAGE', 'FILE_STORAGE', 'OBJECT_STORAGE', 'QUAIS_DOS_SEGUINTES_FORMATOS_PARA_ARMAZENAMENTO_E_GESTAO_DE_DADOS_LHE_SAO_PRIORITARIOS', 'DATA_WAREHOUSE', 'DATA_LAKE', 'DATA_LAKEHOUSE', 'DATA_FABRIC', 'DATA_MESH', 'QUAIS_DAS_SEGUINTES_ARQUITETURAS_PARA_ARMAZENAMENTO_E_GESTAO_DE_DADOS_LHE_SAO_PRIORITARIAS', 'APACHE_HADOOP_HDFS', 'GLUSTERFS', 'JUICEFS', 'MINIO', 'CEPH', 'GARAGE', 'APACHE_OZONE', 'APACHE_HUDI', 'APACHE_ICEBERG', 'DELTA_LAKE', 'DATABRICKS', 'SNOWFLAKE', 'DREMIO', 'STARBURST', 'OUTRO5', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_PARA_ARMAZENAMENTO_E_GESTAO_DE_DADOS_LHE_SAO_PRIORITARIOS', 'OPENMETADATA', 'APACHE_ATLAS', 'AMUNDSEN', 'DATAHUB', 'MARQUEZ', 'CKAN', 'ATLAN', 'ALATION', 'COLLIBRA', 'IMMUTA', 'HIVE_METASTORE', 'UNITY_CATALOG', 'POLARIS_CATALOG', 'OUTRO3', 'SE_

In [46]:
grupos = {

    "FORMATO_ARMAZENAMENTO": [
        "BLOCK_STORAGE",
        "FILE_STORAGE",
        "OBJECT_STORAGE"
    ],

    "FORMATO_ARMAZENAMENTO_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FORMATOS_PARA_ARMAZENAMENTO_E_GESTAO_DE_DADOS_LHE_SAO_PRIORITARIOS"
    ],

    "ARQUITETURA_DADOS": [
        "DATA_WAREHOUSE",
        "DATA_LAKE",
        "DATA_LAKEHOUSE",
        "DATA_FABRIC",
        "DATA_MESH"
    ],

    "ARQUITETURA_DADOS_PRIORITARIO": [
        "QUAIS_DAS_SEGUINTES_ARQUITETURAS_PARA_ARMAZENAMENTO_E_GESTAO_DE_DADOS_LHE_SAO_PRIORITARIAS"
    ],

    "ARMAZENAMENTO_E_GESTAO_DADOS": [
        "APACHE_HADOOP_HDFS",
        "GLUSTERFS",
        "JUICEFS",
        "MINIO",
        "CEPH",
        "GARAGE",
        "APACHE_OZONE",
        "APACHE_HUDI",
        "APACHE_ICEBERG",
        "DELTA_LAKE",
        "DATABRICKS",
        "SNOWFLAKE",
        "DREMIO",
        "STARBURST",
        "OUTRO5"
    ],

    "ARMAZENAMENTO_E_GESTAO_DADOS_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "ARMAZENAMENTO_E_GESTAO_DADOS_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_PARA_ARMAZENAMENTO_E_GESTAO_DE_DADOS_LHE_SAO_PRIORITARIOS"
    ],

    "GOVERNANCA_GESTAO_DADOS": [
        "OPENMETADATA",
        "APACHE_ATLAS",
        "AMUNDSEN",
        "DATAHUB",
        "MARQUEZ",
        "CKAN",
        "ATLAN",
        "ALATION",
        "COLLIBRA",
        "IMMUTA",
        "HIVE_METASTORE",
        "UNITY_CATALOG",
        "POLARIS_CATALOG",
        "OUTRO3"
    ],

    "GOVERNANCA_GESTAO_DADOS_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_GOVERNANCA_E_GESTAO_DE_DADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "GOVERNANCA_GESTAO_DADOS_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_GOVERNANCA_E_GESTAO_DE_DADOS_LHE_SAO_PRIORITARIOS"
    ],

    "SEGURANCA_DADOS": [
        "HASHICORP_VAULT",
        "AKEYLESS",
        "DOPPLER",
        "INFISICAL",
        "OPENBAO",
        "KEYCLOAK",
        "KERBEROS",
        "OKTA",
        "AUTHENTIK",
        "OPA_OPEN_POLICY_AGENT",
        "KYVERNO",
        "CERBOS",
        "APACHE_RANGER",
        "APACHE_KNOX",
        "SONAR_QUBE",
        "OWASP",
        "SNYK",
        "GREMLIN",
        "OUTRO6"
    ],

    "SEGURANCA_DADOS_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_SEGURANCA_DE_DADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "SEGURANCA_DADOS_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_SEGURANCA_DE_DADOS_LHE_SAO_PRIORITARIOS"
    ],

    "ORQUESTRACAO_DADOS": [
        "APACHE_AIRFLOW",
        "PREFECT",
        "DAGSTER",
        "LUIGI",
        "FLYTE",
        "KESTRA",
        "MAGE",
        "MAESTRO",
        "MLRUN",
        "METAFLOW",
        "KEDRO",
        "ARGO_WORKFLOWS",
        "JENKINS",
        "OUTRO4"
    ],

    "ORQUESTRACAO_DADOS_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_ORQUESTRACAO_DE_DADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "ORQUESTRACAO_DADOS_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_ORQUESTRACAO_DE_DADOS_LHE_SAO_PRIORITARIOS"
    ],

    "CONFORMIDADE_DADOS": [
        "MONTE_CARLO",
        "SODA",
        "GREAT_EXPECTATIONS",
        "PANDERA",
        "METAPLANE",
        "BIGEYE",
        "APACHE_GRIFFIN",
        "ANOMALO",
        "DATAFOLD",
        "KENSU",
        "ACCELDATA",
        "LIGHTUP",
        "OUTRO"
    ],

    "CONFORMIDADE_DADOS_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_CONFORMIDADE_DE_DADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "CONFORMIDADE_DADOS_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_CONFORMIDADE_DE_DADOS_LHE_SAO_PRIORITARIOS"
    ],

    "MONITORAMENTO_DADOS": [
        "ZABBIX",
        "OPENTELEMETRY",
        "PROMETHEUS",
        "GRAFANA",
        "JAEGER",
        "ELK_STACK_ELASTIC_LOGSTASH_KIBANA",
        "GRAPHITE",
        "FLUENTD",
        "SPLUNK",
        "NEW_RELIC",
        "DATADOG",
        "AMBARI",
        "OUTRO2"
    ],

    "MONITORAMENTO_DADOS_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_MONITORAMENTO_DE_DADOSRECURSOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "MONITORAMENTO_DADOS_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_MONITORAMENTO_DE_DADOSRECURSOS_LHE_SAO_PRIORITARIOS"
    ]
}


id_cols = ["ID", "EMAIL", "NAME"]

dfs = []

# grupos.items() -> it will return a view object that displays a list of a dictionary's key-value tuple pairs.
# dict_items([('SISTEMA_OPERACIONAL', ['WINDOWS_NT_SERVER', 'WINDOWS_1011', 'LINUX_DEBIAN_UBUNTU_MINT_KALI', 'LINUX_A....

for categoria, cols in grupos.items():

    df_temp = table3.melt(
        id_vars=id_cols,
        value_vars=cols,
        var_name="TECNOLOGIA",
        value_name="RESPOSTA"
    )

    df_temp["CATEGORIA"] = categoria

    dfs.append(df_temp)

df_final = pd.concat(dfs, ignore_index=True)

# Replacing Regex r"OUTRO\d+"
df_final["TECNOLOGIA"] = df_final["TECNOLOGIA"].str.replace(r"OUTRO\d+", "OUTRO", regex=True)

In [47]:
#df_final

In [48]:
resultado = (
    table1
    .merge(
        df_final.drop(columns=["ID","EMAIL"], errors="ignore"),
        on="NAME",
        how="left"
    )
)

# Salvando de forma básica

resultado.to_csv(f'doc_3.csv', index=False)
print(f"Arquivo salvo como doc_3.csv")

Arquivo salvo como doc_3.csv


# Analise relacionado a documento 2

In [7]:
import sqlite3
import pandas as pd
from pathlib import Path

# Dynamic path: go up one level from scripts/ to reach the project root, then into files/
# db_path = Path(__file__).parent.parent / "files" / "cepel_data.db"

# In a Jupyter notebook, use this instead:
db_path = Path.cwd().parent / "scripts" / "cepel_data.db"
conn = sqlite3.connect(str(db_path))

# List all tables in the SQLite database
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)
print(tables)

                                                                         name
0     Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação
1             Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_1_Geral
2          Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_2_Avançada
3  Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps


In [50]:
# Carrega um recorte da tabela de identificação dos respondentes
table2 = pd.read_sql_query('''SELECT * FROM Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_1_Geral''', conn)
table2.columns

Index(['ID', 'START_TIME', 'COMPLETION_TIME', 'EMAIL', 'NAME',
       'LAST_MODIFIED_TIME',
       'ESTRUTURADOS_TABULARES_CSV_PLANILHAS_DATAFRAMES_SQL_DUMPS',
       'SEMIESTRUTURADOS_SEM_ESQUEMA_RIGIDO_JSON_TOON_XML_PARQUET_ORC_AVRO',
       'NAOESTRUTURADOS_SEM_FORMATO_DEFINIDO_TEXTOSDOCS_LOGS_AUDIOS_IMAGENS_VIDEOS_BINARIOS',
       'QUAIS_DOS_SEGUINTES_TIPOS_DE_DADOS_LHE_SAO_PRIORITARIOS',
       ...
       'STREAMLIT', 'DASH', 'SHINY', 'GRADIO', 'PANEL', 'MERCURY', 'EVIDENCE',
       'OUTRO2',
       'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_ANALISE_E_VISUALIZACAO_DE_DADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
       'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_ANALISE_E_VISUALIZACAO_DE_DADOS_LHE_SAO_PRIORITARIOS'],
      dtype='str', length=140)

In [51]:
print(table2.columns.tolist())

['ID', 'START_TIME', 'COMPLETION_TIME', 'EMAIL', 'NAME', 'LAST_MODIFIED_TIME', 'ESTRUTURADOS_TABULARES_CSV_PLANILHAS_DATAFRAMES_SQL_DUMPS', 'SEMIESTRUTURADOS_SEM_ESQUEMA_RIGIDO_JSON_TOON_XML_PARQUET_ORC_AVRO', 'NAOESTRUTURADOS_SEM_FORMATO_DEFINIDO_TEXTOSDOCS_LOGS_AUDIOS_IMAGENS_VIDEOS_BINARIOS', 'QUAIS_DOS_SEGUINTES_TIPOS_DE_DADOS_LHE_SAO_PRIORITARIOS', 'ARQUIVOS_DE_DADOS_XLS_CSV_JSON_PARQUET_IMAGEM', 'SISTEMAS_CORPORATIVOS_ERP_CRM', 'APIS_E_SERVICOS_WEB', 'DADOS_DE_SENSORES_IOT', 'LOGS_DE_SERVIDORES_E_APLICACOES', 'DADOS_DE_REDES_SOCIAIS', 'DADOS_GEOESPACIAIS', 'BANCOS_DE_DADOS_SQL_E_NOSQL', 'OUTRA', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRA_FONTE_DE_DADO_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DAS_SEGUINTES_FONTES_DE_DADOS_LHE_SAO_PRIORITARIAS', 'AIRBYTE', 'FIVETRAN', 'MELTANO', 'APACHE_NIFI', 'APACHE_CAMEL', 'APACHE_KAFKA', 'PULSAR', 'CONFLUENT', 'RABBITMQ', 'APACHE_ROCKETMQ', 'REDPANDA', 'DLT', 'OUTRO3', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_Q

In [52]:
grupos = {

    "TIPO_DADOS": [
        "ESTRUTURADOS_TABULARES_CSV_PLANILHAS_DATAFRAMES_SQL_DUMPS",
        "SEMIESTRUTURADOS_SEM_ESQUEMA_RIGIDO_JSON_TOON_XML_PARQUET_ORC_AVRO",
        "NAOESTRUTURADOS_SEM_FORMATO_DEFINIDO_TEXTOSDOCS_LOGS_AUDIOS_IMAGENS_VIDEOS_BINARIOS"
    ],

    "TIPO_DADOS_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_TIPOS_DE_DADOS_LHE_SAO_PRIORITARIOS"
    ],

    "FONTE_DADOS": [
        "ARQUIVOS_DE_DADOS_XLS_CSV_JSON_PARQUET_IMAGEM",
        "SISTEMAS_CORPORATIVOS_ERP_CRM",
        "APIS_E_SERVICOS_WEB",
        "DADOS_DE_SENSORES_IOT",
        "LOGS_DE_SERVIDORES_E_APLICACOES",
        "DADOS_DE_REDES_SOCIAIS",
        "DADOS_GEOESPACIAIS",
        "BANCOS_DE_DADOS_SQL_E_NOSQL",
        "OUTRA"
    ],

    "FONTE_DADOS_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRA_FONTE_DE_DADO_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "FONTE_DADOS_PRIORITARIO": [
        "QUAIS_DAS_SEGUINTES_FONTES_DE_DADOS_LHE_SAO_PRIORITARIAS"
    ],

    "INGESTAO_DADOS": [
        "AIRBYTE",
        "FIVETRAN",
        "MELTANO",
        "APACHE_NIFI",
        "APACHE_CAMEL",
        "APACHE_KAFKA",
        "PULSAR",
        "CONFLUENT",
        "RABBITMQ",
        "APACHE_ROCKETMQ",
        "REDPANDA",
        "DLT",
        "OUTRO3"
    ],

    "INGESTAO_DADOS_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_INGESTAO_DE_DADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "INGESTAO_DADOS_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_INGESTAO_DE_DADOS_LHE_SAO_PRIORITARIOS"
    ],

    "BANCO_DADOS_SQL": [
        "SQLITE",
        "MYSQL",
        "MARIADB",
        "SQL_SERVER",
        "POSTGRESQL",
        "SUPABASE",
        "ORACLE_DATABASE",
        "IBM_DB2",
        "TIDB",
        "COCKROACHDB",
        "H2_DATABASE",
        "OUTRO4"
    ],

    "BANCO_DADOS_SQL_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_BANCO_DE_DADOS_SQL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "BANCO_DADOS_SQL_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_BANCOS_DE_DADOS_SQL_LHE_SAO_PRIORITARIOS"
    ],

    "BANCO_DADOS_NOSQL": [
        "MONGODB",
        "APACHE_COUCHDB",
        "COUCHBASE",
        "RAVENDB",
        "REDIS",
        "AEROSPIKE",
        "APACHE_HBASE",
        "APACHE_CASSANDRA",
        "SCYLLADB",
        "NEO4J",
        "ORIENTDB",
        "ARANGODB",
        "OUTRO5"
    ],

    "BANCO_DADOS_NOSQL_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_BANCO_DE_DADOS_NOSQL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "BANCO_DADOS_NOSQL_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_BANCOS_DE_DADOS_NOSQL_LHE_SAO_PRIORITARIOS"
    ],

    "BANCO_DADOS_OLAP": [
        "ELASTICSEARCH",
        "APACHE_SOLR",
        "OPENSEARCH",
        "INFLUXDB",
        "IOTDB",
        "QUESTDB",
        "TIMESCALE",
        "DUCKDB",
        "APACHE_DATAFUSION",
        "CLICKHOUSE",
        "APACHE_PINOT",
        "APACHE_DRUID",
        "APACHE_DORIS",
        "STARROCKS",
        "OUTRO"
    ],

    "BANCO_DADOS_OLAP_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_BANCO_DE_DADOS_OLAP_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "BANCO_DADOS_OLAP_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_BANCOS_DE_DADOS_OLAP_LHE_SAO_PRIORITARIOS"
    ],

    "BANCO_DADOS_VETORIAL": [
        "PINECONE",
        "MILVUS",
        "QDRANT",
        "CHROMA",
        "WEAVIATE",
        "VESPA",
        "VALD",
        "LANCEDB",
        "DEEPLAKE",
        "MARQO",
        "POSTGRESQL_PGVECTOR",
        "MONGODB_VECTOR",
        "ELASTICSEARCH_VECTOR",
        "REDIS_VECTOR",
        "OUTRO6"
    ],

    "BANCO_DADOS_VETORIAL_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_BANCO_DE_DADOS_VETORIAL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "BANCO_DADOS_VETORIAL_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_BANCOS_DE_DADOS_VETORIAIS_LHE_SAO_PRIORITARIOS"
    ],

    "TRANSFORMACAO_DADOS": [
        "APACHE_SPARK",
        "APACHE_BEAM",
        "APACHE_FLINK",
        "APACHE_SAMZA",
        "APACHE_STORM",
        "APACHE_ARROW",
        "PYSPARK",
        "POLARS",
        "RAPIDS_CUDF",
        "RAY",
        "DASK",
        "DUCKDB2",
        "PANDAS",
        "DBT",
        "PENTAHO",
        "TALEND",
        "OUTRO7"
    ],

    "TRANSFORMACAO_DADOS_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_TRANSFORMACAO_DE_DADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "TRANSFORMACAO_DADOS_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_TRANSFORMACAO_DE_DADOS_LHE_SAO_PRIORITARIOS"
    ],

    "ANALISE_VISUALIZACAO_DADOS": [
        "TRINO",
        "STARBURST",
        "PRESTO",
        "DREMIO",
        "APACHE_DRILL",
        "APACHE_IMPALA",
        "APACHE_SUPERSET",
        "METABASE",
        "POWERBI",
        "TABLEAU",
        "QLIK",
        "LOOKER",
        "STREAMLIT",
        "DASH",
        "SHINY",
        "GRADIO",
        "PANEL",
        "MERCURY",
        "EVIDENCE",
        "OUTRO2"
    ],

    "ANALISE_VISUALIZACAO_DADOS_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_ANALISE_E_VISUALIZACAO_DE_DADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "ANALISE_VISUALIZACAO_DADOS_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_ANALISE_E_VISUALIZACAO_DE_DADOS_LHE_SAO_PRIORITARIOS"
    ]
}


id_cols = ["ID", "EMAIL", "NAME"]

dfs = []

# grupos.items() -> it will return a view object that displays a list of a dictionary's key-value tuple pairs.
# dict_items([('SISTEMA_OPERACIONAL', ['WINDOWS_NT_SERVER', 'WINDOWS_1011', 'LINUX_DEBIAN_UBUNTU_MINT_KALI', 'LINUX_A....

for categoria, cols in grupos.items():

    df_temp = table2.melt(
        id_vars=id_cols,
        value_vars=cols,
        var_name="TECNOLOGIA",
        value_name="RESPOSTA"
    )

    df_temp["CATEGORIA"] = categoria

    dfs.append(df_temp)

df_final = pd.concat(dfs, ignore_index=True)

# Replacing Regex r"OUTRO\d+"
df_final["TECNOLOGIA"] = df_final["TECNOLOGIA"].str.replace(r"OUTRO\d+", "OUTRO", regex=True)

In [53]:
resultado = (
    table1
    .merge(
        df_final.drop(columns=["ID","EMAIL"], errors="ignore"),
        on="NAME",
        how="left"
    )
)

# Salvando de forma básica

resultado.to_csv(f'doc_2.csv', index=False)
print(f"Arquivo salvo como doc_2.csv")

Arquivo salvo como doc_2.csv
